### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [1]:
import polars.selectors as cs
import polars as pl
import pyarrow
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path
folder= Path().resolve().parent /'input'
meta_data_folder=Path().resolve().parent / 'metadata'

### 1. Load multiple dataset for different buildings.

In [3]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / '*')

In [4]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [5]:
%%time
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
Meta_data=pl.scan_parquet(meta_data_folder/'MetaData.parquet').with_columns(
    pl.col('bldg_id').cast(pl.UInt32)).filter(
    pl.col('bldg_id').is_in(bldgs
                           )).select(pl.col('in.occupants').cast(pl.Int32),'in.state','in.county',
                        'in.representative_income','in.area_median_income','in.income',
                        'in.income_recs_2020','in.income_recs_2015',
                        'in.building_america_climate_zone','in.ashrae_iecc_climate_zone_2004_sub_cz_split',
                        pl.col('in.bedrooms').cast(pl.Int32),'in.tenure','in.household_has_tribal_persons','bldg_id').unique()

CPU times: user 909 μs, sys: 1.82 ms, total: 2.73 ms
Wall time: 3.46 ms


In [6]:
%%time
# merge the climate zone changes into the original data
df=data.join(Meta_data, on='bldg_id')

CPU times: user 125 μs, sys: 18 μs, total: 143 μs
Wall time: 154 μs


In [7]:
df=df.rename({'in.ashrae_iecc_climate_zone_2004_sub_cz_split': 'climatezone',
                'out.electricity.cooling.energy_consumption..kwh': 'out.electricity.AC.energy_consumption..kwh'}).with_columns(
            pl.col('timestamp').dt.weekday().alias('day of the week'),
            pl.col('timestamp').dt.hour().alias('hour of the day'),
            pl.col('timestamp').dt.day().alias('day of the month'),
            pl.col('timestamp').dt.ordinal_day().alias('day of the year'),
            pl.col('timestamp').dt.week().alias('week of the year'),
            pl.col('timestamp').dt.month().alias('month of the year'),
            pl.col('timestamp').dt.quarter().alias('quarter')).with_columns(
            pl.when(pl.col('day of the week').is_in([6,7])).then(pl.lit("Yes")).otherwise(pl.lit("No")).alias('IsWeekend')
            ).select(
    cs.matches('^out.electricity.*|^out.site_energy.*|^bldg*|^day*|^hour*|^week*|^month*|^time*|^quarter|^IsWeekend|^in.*|^Short|^climatezone$')
).drop_nulls().collect()


----------------------------------------------

### Label Encoder for categorical variables

In [8]:
from sklearn.preprocessing import LabelEncoder
def encoding_categories(x):
    le=LabelEncoder()
    for dtype,col in zip(x.dtypes,x.columns):
        if dtype==pl.String:
            x=x.with_columns(pl.Series(col, le.fit_transform(x[col])))
    return x

## Preprocess the data

In [9]:
# defining x and y
def prepare_observations(d):
    x=d.select(cs.matches('^day*|^hour*|^week*|^month*|^time*|^quarter|^IsWeekend|^in.*|^Short|^climatezone$').exclude('in.sqft'),
            pl.col('out.electricity.total.energy_consumption..kwh').alias('total_usage')).with_columns(pl.col('timestamp').dt.timestamp())
    y=d.select(cs.matches('^out.electricity.*..kwh$').exclude('out.electricity.total.energy_consumption..kwh'))
    x=encoding_categories(x)
    return x,y

## test standardizing the data with the standard scaler

In [10]:
from sklearn.preprocessing import StandardScaler
def normlize_standard(x,y=None):
    scaler=StandardScaler()
    x=scaler.fit_transform(x)
    if y is not None:
        y=scaler.fit_transform(y)
        return x, y
    else:
        return x

## test standardizing the data with the Min Max Scaler

In [11]:
from sklearn.preprocessing import MinMaxScaler
def normalize_minMax(x, y=None):
    scaler=MinMaxScaler()
    x=scaler.fit_transform(x)
    if y is not None:
        y=scaler.fit_transform(y)
        return x,y 
    else:
        return x

### Prepare cross-validation model


In [12]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
def model(mod, x_train, x_test, y_train, y_test, metric):
    mod.fit(x_train, y_train)
    predicted=mod.predict(x_test)
    return metric(y_test, predicted)
        # r2_score(y_test, predicted)
        # mean_absolute_error(y_test, predicted), 
        # mean_squared_error(y_test, predicted), 
        # root_mean_squared_error(y_test, predicted)

In [13]:
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import shuffle
import xgboost as xg
def cross_folding(x,y, metric):
    kf=KFold(n_splits=5, shuffle=True)
    arr=[]
    lf=pd.DataFrame()
    for i, (train,test) in enumerate(kf.split(x,y)):
        x_train, x_test, y_train, y_test=x[train], x[test], y[train], y[test]
        LR=model(LinearRegression(), x_train, x_test, y_train, y_test, metric)
        RF=model(RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1), x_train, x_test, y_train, y_test, metric)
        XG=model(xg.XGBRegressor(learning_rate= 0.1, max_depth= 7, subsample= 0.7), x_train, x_test, y_train, y_test, metric)
        frame=pd.DataFrame({
        "Split Number": [i],
        "Linear Regression": [LR],
        "Random Forest": [RF],
        "XG Boost": [XG]
        })
        arr.append(frame)
    lf=pd.concat(arr)
    return lf.mean()

In [14]:
def without_preprocessing(d):
    x,y=prepare_observations(d)
    average=cross_folding(x, y, r2_score)
    return average

In [15]:
def with_standardScalar(d):
    x,y=prepare_observations(d)
    x=normlize_standard(x)
    average=cross_folding(x, y, r2_score)
    return average

In [16]:
def with_minMaxScalar(d):
    x,y=prepare_observations(d)
    x=normalize_minMax(x)
    average=cross_folding(x, y, r2_score)
    return average

In [17]:
def apply_modelling(d):
    average_0=without_preprocessing(d)
    average_1=with_standardScalar(d)
    average_2=with_minMaxScalar(d)   
    return pd.concat([average_0, average_1, average_2], axis=1, keys=['no normalization','standard Scalar','MinMaxScalar'])

In [ ]:
apply_modelling(df)

---------------------

### Tuning XGboost Model

In [ ]:
from sklearn.model_selection import GridSearchCV
def tunning(training_data, testing_data, model, param):
        x_train, x_test, y_train, y_test=train_test_split(x,y, test_size=0.3)
        grid_search = GridSearchCV(model, param_grid=param, cv=KFold(3), 
                                   scoring='r2',
                                   n_jobs=-1)
        grid_search.fit(x_train, y_train)
        return grid_search.best_params_, grid_search.best_score_

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
params= {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01, 0.001],
    'subsample': [0.5, 0.7, 1]
}
x,y=prepare_observations(df)
best_param, best_score=tunning(x,y, XGBRegressor(), params)

In [ ]:
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.3, random_state=1)
XG=model(XGBRegressor(learning_rate= 0.1, max_depth= 7, subsample= 0.7), x_train, x_test, y_train, y_test, r2_score)
XG

### Deep Learning with pytorch

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as tf

In [13]:
import gc
torch.cuda.empty_cache()
gc.collect()

4315

In [14]:
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, features_num=23, h1=23, out_features=29):
        super().__init__()
        self.lstm = nn.LSTM(features_num, h1, 1, batch_first=True)
        self.conv = nn.Conv1d(h1, h1, kernel_size=7)
        self.fc = nn.Linear(h1, out_features)

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x.permute(0, 2, 1)
        x = self.conv(x)  
        x = x.permute(0, 2, 1)  
        x = self.fc(x)  # 

        return torch.relu(x)

In [21]:
from sklearn.model_selection import train_test_split
model=Model()
df=df.group_by('bldg_id', 'timestamp').agg(pl.all())
df

bldg_id,timestamp,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.AC.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,out.electricity.heating.energy_consumption..kwh,out.electricity.heating_fans_pumps.energy_consumption..kwh,out.electricity.heating_hp_bkup.energy_consumption..kwh,out.electricity.heating_hp_bkup_fa.energy_consumption..kwh,out.electricity.hot_water.energy_consumption..kwh,out.electricity.hot_water_solar_th.energy_consumption..kwh,out.electricity.lighting_exterior.energy_consumption..kwh,out.electricity.lighting_garage.energy_consumption..kwh,out.electricity.lighting_interior.energy_consumption..kwh,out.electricity.mech_vent.energy_consumption..kwh,out.electricity.net.energy_consumption..kwh,out.electricity.permanent_spa_heat.energy_consumption..kwh,out.electricity.permanent_spa_pump.energy_consumption..kwh,out.electricity.plug_loads.energy_consumption..kwh,out.electricity.pool_heater.energy_consumption..kwh,out.electricity.pool_pump.energy_consumption..kwh,out.electricity.pv.energy_consumption..kwh,out.electricity.range_oven.energy_consumption..kwh,out.electricity.refrigerator.energy_consumption..kwh,out.electricity.television.energy_consumption..kwh,out.electricity.total.energy_consumption..kwh,out.electricity.well_pump.energy_consumption..kwh,out.site_energy.net.energy_consumption..kwh,out.site_energy.total.energy_consumption..kwh,out.electricity.ceiling_fan.energy_consumption_intensity..kwh_per_ft2,out.electricity.clothes_dryer.energy_consumption_intensity..kwh_per_ft2,…,out.electricity.lighting_interior.energy_consumption_intensity..kwh_per_ft2,out.electricity.mech_vent.energy_consumption_intensity..kwh_per_ft2,out.electricity.net.energy_consumption_intensity..kwh_per_ft2,out.electricity.permanent_spa_heat.energy_consumption_intensity..kwh_per_ft2,out.electricity.permanent_spa_pump.energy_consumption_intensity..kwh_per_ft2,out.electricity.plug_loads.energy_consumption_intensity..kwh_per_ft2,out.electricity.pool_heater.energy_consumption_intensity..kwh_per_ft2,out.electricity.pool_pump.energy_consumption_intensity..kwh_per_ft2,out.electricity.pv.energy_consumption_intensity..kwh_per_ft2,out.electricity.range_oven.energy_consumption_intensity..kwh_per_ft2,out.electricity.refrigerator.energy_consumption_intensity..kwh_per_ft2,out.electricity.television.energy_consumption_intensity..kwh_per_ft2,out.electricity.total.energy_consumption_intensity..kwh_per_ft2,out.electricity.well_pump.energy_consumption_intensity..kwh_per_ft2,out.site_energy.net.energy_consumption_intensity..kwh_per_ft2,out.site_energy.total.energy_consumption_intensity..kwh_per_ft2,in.occupants,in.state,in.county,in.representative_income,in.area_median_income,in.income,in.income_recs_2020,in.income_recs_2015,in.building_america_climate_zone,climatezone,in.bedrooms,in.tenure,in.household_has_tribal_persons,day of the week,hour of the day,day of the month,day of the year,week of the year,month of the year,quarter,IsWeekend
u32,datetime[ms],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],…,list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],list[list[f32]],

In [20]:
x,y=prepare_observations(df)
x,y=normlize_standard(x, y)
x_train, x_test, y_train, y_test=train_test_split(x, y, random_state=42, test_size=0.3)
x_train=torch.from_numpy(x_train).float()
x_test=torch.from_numpy(x_test).float()
y_train=torch.from_numpy(y_train).float()
y_test=torch.from_numpy(y_test).float()

ValueError: setting an array element with a sequence.

In [16]:
## Creating a criterion to measure the error of the model
criterion=nn.MSELoss()

In [17]:
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [18]:
# from torch.utils.data import TensorDataset, DataLoader
# data_set=TensorDataset(x_train, y_train)
# dataset=DataLoader(dataset=data_set, batch_size=2048, shuffle=True)

In [19]:
%%time
epochs=30
losses=[]
for i in range(epochs):
    # for x, y in dataset:
    y_pred=model.forward(x_train)
    loss=criterion(y_pred, y_train)
    losses.append(loss)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if i %10 ==0:
        print(f"{i} iteration, loss: {loss}")

CPU times: user 610 ms, sys: 165 ms, total: 775 ms
Wall time: 456 ms


RuntimeError: permute(sparse_coo): number of dimensions in the tensor input does not match the length of the desired ordering of dimensions i.e. input.dim() = 2 is not equal to len(dims) = 3

## Calculating measures

In [ ]:
# prepare measurement
Test_data=x.head(1)
Actual_data=y.row(1)

In [ ]:
Test_data

In [ ]:
Actual_data

In [ ]:
# Select the best model 
import xgboost as xg
from sklearn.model_selection import train_test_split
x,x_test, y, y_test= train_test_split(x,y, test_size=.3, random_state=42)
xg_model=xg.XGBRegressor()
xg_model.fit(x,y)
estimated_values=xg_model.predict(Test_data)

In [ ]:
estimated_values.flatten()

## Observation

Apparently using r square as a metric to represent to the data is a misrepresentation to the model's performance.
R square formula is - >1- ( total of squared residulas /total of squared standard deviation)
Knowing MAE is the total of residuals -> total(y - yPredicted)
giving r square = 0.02 therefore
giving standard deviation of .001 as given above
then r square= .0004 / .00001= 40 -> 1-40 =-39 which is wrong

Therefore, MAE score is the perfect representation of the data.
another testing metric should be provided.

Input data = timestamp Total      Usage     Climate Zone.
             1545212700000000     1.22827       1

Actual result=(0.1280200034379959, 0.0, 0.0, 0.01245999988168478, 0.0).


Estimated result= [2.9516332e-02, 4.0768548e-03, 1.0928856e-17, 1.0773698e-02,
       2.7552096e-03].